# 04 — Explore & Debug
Interactive querying — try any combination of chunking strategy, retrieval method,
and metadata filters. Use this to understand WHY a config performs the way it does.

In [ ]:
# ── Environment & path setup ─────────────────────────────────────────────────
import os, sys
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd().parent  # notebooks/ -> project root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
QDRANT_URL     = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")

print("OPENAI_API_KEY set:", bool(OPENAI_API_KEY))
print("QDRANT_URL       :", QDRANT_URL)

In [ ]:
# ── Clients ──────────────────────────────────────────────────────────────────
from openai import OpenAI
from qdrant_client import QdrantClient
from fastembed import SparseTextEmbedding

oai        = OpenAI(api_key=OPENAI_API_KEY)
qdrant     = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY or None)
bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
print("Ready.")

In [ ]:
# ── Try any config here ───────────────────────────────────────────────────────
from pipeline.rag import rag

result = rag(
    query    = "What are an employee's rights when requesting flexible work?",

    # ← swap these to compare:
    chunking  = 'recursive',   # 'recursive' | 'semantic' | 'header' | 'parent_child'
    strategy  = 'hybrid',      # 'dense' | 'hybrid' | 'multi_query' | 'compression'

    oai=oai, qdrant=qdrant, bm25_model=bm25_model,
    openai_api_key=OPENAI_API_KEY,

    # Optional metadata filters:
    category       = 'flexible_work',
    role_relevance = 'employee',
)

print('Answer:')
print(result['answer'])
print(f"\nChunking: {result['chunking_strategy']}  |  Retrieval: {result['retrieval_strategy']}")
print('\nChunks used:')
for c in result['chunks']:
    print(f"  [{c['rank']}] score={c['score']}  {c['source']} — {c['title']}")
    print(f"       {c['text'][:120]}...")

In [ ]:
# ── Side-by-side comparison of two configs ────────────────────────────────────
QUERY = "What should employers do when an employee shows signs of burnout?"

configs = [
    dict(chunking='recursive', strategy='dense'),
    dict(chunking='semantic',  strategy='hybrid'),
]

for cfg in configs:
    out = rag(
        query=QUERY,
        oai=oai, qdrant=qdrant, bm25_model=bm25_model,
        openai_api_key=OPENAI_API_KEY,
        **cfg,
    )
    print(f"\n{'='*60}")
    print(f"  {cfg['chunking']} / {cfg['strategy']}")
    print(f"{'='*60}")
    print(out['answer'])